# Deploy a fine-tuned LLM on a SageMaker AI endpoint

This notebook hosts a model you have stored in S3 (in Hugging Face format) as a live inference API on a SageMaker real-time endpoint, using the vLLM serving container. The example is a fine-tuned Qwen2.5-32B, but the steps are the same for any Hugging Face model: you only change the machine size and the S3 path.

Run the cells top to bottom. Each is preceded by a short note explaining what it does and what, if anything, you need to change. For the background (permissions, machine sizing, and troubleshooting) see the SageMaker AI page of the migration guide.

## Before you start

Make sure you have:

- Your model weights in an S3 bucket, in one clean folder (config, `*.safetensors`, tokenizer, nothing else).
- A SageMaker **execution role** whose permissions include read access to that bucket. If your bucket name does not contain the word "sagemaker", the default policy will not grant this and deployment fails with a 403; see the guide's permissions section.
- GPU quota for the instance type you plan to use, in the region you are working in.

You are running this notebook inside SageMaker, so it already uses your AWS identity: you never paste AWS keys.

## 1. Set up

Install the AWS SDK (`boto3`) and connect. Run the next three cells as they are: they install the library, open a session, and define two small helpers (one to look up your role, one to poll the endpoint while it starts). If Jupyter asks you to restart the kernel after the install, do so, then continue.

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import time
import re
import json
import boto3
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")  # client to intreract with SageMaker
sm_runtime = boto3.client("sagemaker-runtime")  # client to intreract with SageMaker Endpoints

In [ ]:
#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    sts = boto3.client('sts')
    response = sts.get_caller_identity()
    assumed_role = response['Arn']
    role = re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", assumed_role)
    return role


def wait_for_endpoint(endpoint_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{endpoint_name}': "
    print(progress)
    
    status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
    
    while status == "Creating":
        time.sleep(sleep_time)
        
        status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        
        clear_output(wait=True)
        progress += ind
        print(progress)
  
    print(f"Endpoint: '{endpoint_name}', Status: '{status}'")


def wait_for_ic(ic_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{ic_name}': "
    print(progress)
    
    status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
    
    while status == "Creating":
        time.sleep(sleep_time)
        
        status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
        
        clear_output(wait=True)
        progress += ind
        print(progress)
  
    print(f"IC: '{ic_name}', Status: '{status}'")

## 2. Your execution role

The endpoint runs *as* an IAM role, and that role is what reads your weights from S3. Set it in the next cell by replacing the ARN with your own execution role. You can find the ARN in the AWS Console under **IAM → Roles**, or replace the value with `get_sagemaker_role()` to use the role this notebook is already running as.

Permissions cannot be fixed from here: a role is not allowed to edit its own permissions, so any change (such as granting bucket access) must be made by an administrator in the IAM console.

In [ ]:
role = get_sagemaker_role()
# Or if you are not running this in SageMaker, hardcode the role ARN:
# role = "arn:aws:iam::111122223333:role/service-role/AmazonSageMaker-ExecutionRole-XXXXXXXXXXXXXXX"
print(role)

## 3. Point at your model and choose the machine

Edit the two values in the next cell:

- `instance` is the GPU machine and how many GPUs it has. The example uses `ml.g7e.12xlarge` (2 GPUs), which suits a 32B model. Pick yours from the sizing table on the guide page, and keep `num_gpu` equal to the number of GPUs on the instance.
- `model_data_s3` is the S3 folder holding your weights. It **must end with a slash**.

The remaining values (the generated names and the startup timeout) can stay as they are.

In [ ]:
instance = {"type": "ml.g7e.12xlarge", "num_gpu": 2}
model_data_s3 = "s3://YOUR-BUCKET/YOUR-MODEL-FOLDER/"   # must end with /
model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 900
variant_name = "v1"

### The serving container

SageMaker loads your model into a ready-made serving container. This notebook uses **vLLM** (an AWS Deep Learning Container), a fast and widely used engine. Run the next cell as it is; it sets the container image and its configuration.

Two values are worth knowing:

- `SM_VLLM_MODEL` is `/opt/ml/model`, the folder inside the machine where SageMaker automatically places the weights it downloads from your S3 path. Leave it as is.
- `SM_VLLM_MAX_MODEL_LEN` is the longest conversation the model will accept, and reserving space for it uses GPU memory. If a model is a tight fit and you hit out-of-memory errors, lower it (for example to `16384`).

In [ ]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:0.19.0-gpu-py312-cu129-ubuntu22.04-sagemaker"

env = {
    "SM_VLLM_MODEL": "/opt/ml/model",                # where SageMaker puts your S3 weights
    "SM_VLLM_SERVED_MODEL_NAME": "qwen2.5-ft",       # name used in the API
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    "SM_VLLM_MAX_MODEL_LEN": "32768",
    "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
    "SM_VLLM_TOOL_CALL_PARSER": "hermes",            # Qwen2.5 uses hermes-style tool calls
}

## 4. Deploy

Run the next two cells in order:

1. The first registers your model, linking the container, its configuration, and your S3 weights under a single name.
2. The second creates the endpoint configuration (the machine to use) and the endpoint itself. Creating the endpoint is what switches the GPU on and starts the hourly charge. The cell then waits, printing a dot about once a minute until the status is `InService`.

Expect several minutes (often 10 to 15 for a large model) while AWS provisions the machine, downloads the weights, and loads them onto the GPUs. When it prints `Status: 'InService'`, the model is live.

In [ ]:
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": model_data_s3,
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
            }
        },
    },
)

In [ ]:
_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
        },
    ],
)

_ = sm.create_endpoint(EndpointName=endpoint_name, 
                       EndpointConfigName=endpoint_config_name)

_ = wait_for_endpoint(endpoint_name)

### Check status and logs

These cells show what is running and the endpoint's detailed status. While it starts the status is `Creating`; then it becomes either `InService` (ready) or `Failed`. If it fails, `describe_endpoint` prints a reason, and the full container logs are in **CloudWatch** under the log group `/aws/sagemaker/Endpoints/<your-endpoint-name>`. The last cell prints the role and its trust policy, which is handy when debugging permissions.

In [ ]:
sm.list_endpoints()["Endpoints"]

In [ ]:
d = sm.describe_endpoint(EndpointName=endpoint_name)
print(d["EndpointStatus"])
print(d.get("FailureReason", "no failure recorded"))


In [ ]:
iam = boto3.client("iam")
role_name = role.split('/')[-1]
r = iam.get_role(RoleName=role_name)
print(r["Role"]["Arn"])
print(r["Role"]["AssumeRolePolicyDocument"])

## 5. Test the endpoint

Send the model a request. A single non-streaming request must complete within **60 seconds**, a hard limit for real-time endpoints, so keep `max_tokens` sensible. For long answers use the streaming cell further down, which is not bound by that limit.

### Sampling settings

Reasonable defaults for a Qwen2.5 instruct model are a temperature of about `0.7` and `top_p` of about `0.8` (the values used in the next cell). Lower the temperature for more deterministic answers, raise it for more variety.

#### One question (non-streaming)

Send a single prompt and print the full reply once it is ready.

In [ ]:
payload = {
    "messages": [
        {"role": "user", "content": "What model are you?"}
    ],
    "max_tokens": 512,
    "temperature": 0.7,
    "top_p": 0.8,
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

#### Streaming (for long answers)

Streaming keeps the connection open and prints tokens as they are generated, so it is not bound by the 60-second limit. Run the helper cell, then the call cell.

In [ ]:
#
# Helper function for streamin invocation
#
import io

import json
import time
import boto3
from IPython.display import clear_output

class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()
            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]
            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise
            if "PayloadPart" not in chunk:
                print("Unknown event type:" + chunk)
                continue
            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])

def stream_response(endpoint_name, inputs, max_tokens=8189, temperature=0.7, top_p=0.9):
    body = {
        "messages": [{"role": "user", "content": [{"type": "text", "text": inputs}]}],
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": top_p,
        "stream": True,
        "stop": ["<|im_end|>", "\nuser", "\nassistant"],
    }

    resp = sm_runtime.invoke_endpoint_with_response_stream(
        EndpointName=endpoint_name,
        Body=json.dumps(body),
        ContentType="application/json",
    )

    full_response = ""
    start_time = time.time()
    token_count = 0

    for line in LineIterator(resp["Body"]):
        if line != b"" and b"{" in line:
            data = json.loads(line[line.find(b"{"):].decode("utf-8"))
            delta = data["choices"][0]["delta"]
            token_text = delta.get("reasoning") or delta.get("content") or ""
            full_response += token_text
            token_count += 1
            print(token_text, end="", flush=True)   # append only, no clear

    tps = token_count / max(time.time() - start_time, 1e-9)
    print(f"\n\nTokens per Second: {tps:.2f}")
    return full_response

In [ ]:
#inputs = "What is greater 9.11 or 9.8?"
inputs = "Solve this problem step by step: What is 17% of 265?"
output = stream_response(endpoint_name, inputs,
                         max_tokens=500)

## 6. Shut it down (do this every time)

**The endpoint bills for every second it is `InService`, used or not, and deleting it is the only thing that stops that charge.** The next cell deletes the three objects this notebook created: the endpoint, its configuration, and the model. The cells after it list what is left so you can confirm nothing is running; the final cell force-deletes every endpoint, config, and model in the region if you have lost track.

The JupyterLab space this notebook runs in also bills, by a smaller amount, while it is open. Stop it from the SageMaker Studio console when you finish for the day.

In [ ]:
_ = sm.delete_endpoint(EndpointName=endpoint_name)
_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
_ = sm.delete_model(ModelName=model_name)

In [ ]:
sm.list_endpoints()["Endpoints"]

In [ ]:
endpoints = sm.list_endpoints(MaxResults=100)["Endpoints"]
for ep in endpoints:
    print(f"{ep['EndpointName']:<40} {ep['EndpointStatus']:<12} {ep['CreationTime']:%Y-%m-%d %H:%M}")

In [ ]:
for ep in sm.list_endpoints(MaxResults=100)["Endpoints"]:
    name = ep["EndpointName"]
    print(f"Deleting endpoint: {name}")
    sm.delete_endpoint(EndpointName=name)

# Endpoint configs are separate objects and don't get deleted with the endpoint
for cfg in sm.list_endpoint_configs(MaxResults=100)["EndpointConfigs"]:
    name = cfg["EndpointConfigName"]
    print(f"Deleting endpoint config: {name}")
    sm.delete_endpoint_config(EndpointConfigName=name)

# Same for model objects (these are just metadata, no cost, but tidy is tidy)
for m in sm.list_models(MaxResults=100)["Models"]:
    name = m["ModelName"]
    print(f"Deleting model: {name}")
    sm.delete_model(ModelName=name)